<a href="https://www.kaggle.com/code/vijay20213/single-lgb-with-0-96656-score?scriptVersionId=328680416" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
FOLDS = 5
ID = 'id'
TARGET = 'class'
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}
TARGET_INV_MAPPING = {
    0:"GALAXY",
    1:"QSO",
    2:"STAR"
}
RUN_OPTUNA = False
RUN_SKF = True
SEED = 42
N_FOLDS = 5

# Data Loading & Original Dataset Preparation

In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")
original = pd.read_csv("/kaggle/input/datasets/vijay20213/stellar-class-prediction/star_classification.csv") 

train_id = train[ID]
test_id = test[ID]

def get_spectral_type(g, r):
    return pd.cut(
        r - g, 
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

def get_galaxy_population(u, r):
    return pd.cut(
        u - r, 
        [-np.inf, 1.4, 2.2, np.inf],
        labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
    ).astype(str)

original['spectral_type'] = get_spectral_type(original['g'], original['r'])
original['galaxy_population'] = get_galaxy_population(original['u'], original['r'])

target_map = {'GALAXY': 'GALAXY', 'QSO': 'QSO', 'STAR': 'STAR'}
original[TARGET] = original[TARGET].map(target_map)
train[TARGET] = train[TARGET].map(target_map)

base_features = [col for col in train.columns if col not in [TARGET, ID]]
cat_cols_init = train.drop(columns=[ID, TARGET]).select_dtypes(include=['object']).columns.tolist()
num_cols_init = [c for c in train.drop(columns=[ID, TARGET]).select_dtypes(exclude=['object']).columns]

original_numeric_target = original[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
original_global_mean = original_numeric_target.mean()
original_global_median = original_numeric_target.median()

original_stats = {}
for col in base_features:
    if col in original.columns:
        orig_temp = original[[col]].copy()
        orig_temp[TARGET] = original_numeric_target
        if col in num_cols_init:
            orig_temp[col] = np.floor(orig_temp[col])
            
        stats = orig_temp.groupby(col)[TARGET].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
        original_stats[col] = stats

# Feature Engineering Pipeline Definition

In [3]:
color_pairs = [('u', 'g'), ('g', 'r'), ('r', 'i'), ('i', 'z'), ('u', 'z')]
important_combos = sorted([
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
    ('g_cat_', 'r_cat_'), 
])

category_map = {}

def feature_engineering(df, fit=False):
    df = df.copy()
    eps = 1e-6

    for col, stats_df in original_stats.items():
        if col in df.columns:
            df_key = df[[col]].copy()
            if col in num_cols_init:
                df_key[col] = np.floor(df_key[col])
                
            merged_stats = df_key.merge(stats_df, on=col, how='left').drop(columns=[col])
            df = pd.concat([df, merged_stats], axis=1)
            
            fill_values = {
                f"orig_{col}_mean": original_global_mean, f"orig_{col}_median": original_global_median,
                f"orig_{col}_std": 0.0, f"orig_{col}_skew": 0.0, f"orig_{col}_count": 0.0
            }
            df.fillna(value=fill_values, inplace=True)
            for s in ['mean', 'median', 'std', 'skew', 'count']:
                df[f"orig_{col}_{s}"] = df[f"orig_{col}_{s}"].astype('float32')

    if 'redshift' in df.columns:
        df['_is_star_redshift'] = (df['redshift'] <= 0).astype('int32')
        df['_dist_modulus_proxy'] = (5 * np.log10(np.abs(df['redshift']) + eps)).astype('float32')
        if 'r' in df.columns: df['_absolute_r_proxy'] = (df['r'] - df['_dist_modulus_proxy']).astype('float32')
        if 'g' in df.columns: df['_absolute_g_proxy'] = (df['g'] - df['_dist_modulus_proxy']).astype('float32')

    bands = ['u', 'g', 'r', 'i', 'z']
    if all(b in df.columns for b in bands):
        df['_ugriz_skew'] = df[bands].skew(axis=1).astype('float32')
        df['_ugriz_kurt'] = df[bands].kurt(axis=1).astype('float32')
        df['_ugriz_mean'] = df[bands].mean(axis=1).astype('float32')
        df['_ugriz_std'] = df[bands].std(axis=1).astype('float32')
        df['_ugriz_max_diff'] = (df[bands].max(axis=1) - df[bands].min(axis=1)).astype('float32')

    if 'alpha' in df.columns and 'delta' in df.columns:
        alpha_rad, delta_rad = np.radians(df['alpha']), np.radians(df['delta'])
        
        df['_coord_x'] = (np.cos(delta_rad) * np.cos(alpha_rad)).astype('float32')
        df['_coord_y'] = (np.cos(delta_rad) * np.sin(alpha_rad)).astype('float32')
        df['_coord_z'] = (np.sin(delta_rad)).astype('float32')
        
        ra_gp = np.radians(192.85948)
        dec_gp = np.radians(27.12825)
        l_cp = np.radians(122.93314)
        
        sin_b = np.sin(delta_rad) * np.sin(dec_gp) + np.cos(delta_rad) * np.cos(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_b'] = np.degrees(np.arcsin(np.clip(sin_b, -1.0, 1.0))).astype('float32')
        
        y_l = np.cos(delta_rad) * np.sin(alpha_rad - ra_gp)
        x_l = np.sin(delta_rad) * np.cos(dec_gp) - np.cos(delta_rad) * np.sin(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_l'] = np.degrees(l_cp - np.arctan2(y_l, x_l)).astype('float32') % 360.0
        
        df['_sin_alpha'] = np.sin(alpha_rad).astype('float32')
        df['_cos_alpha'] = np.cos(alpha_rad).astype('float32')
        df['_sin_delta'] = np.sin(delta_rad).astype('float32')
        df['_cos_delta'] = np.cos(delta_rad).astype('float32')
        
        df['_alpha_plus_delta'] = (df['alpha'] + df['delta']).astype('float32')
        df['_alpha_minus_delta'] = (df['alpha'] - df['delta']).astype('float32')
        df['_alpha_mod1'] = (df['alpha'] % 1).astype('float32')
        df['_delta_mod1'] = (df['delta'] % 1).astype('float32')
        df['_total_spatial_distance'] = np.sqrt(df['alpha']**2 + df['delta']**2).astype('float32')
    
    if 'redshift' in df.columns:
        df['_log_redshift'] = np.log1p(np.abs(df['redshift'])).astype('float32')
        df['_r_x_redshift'] = (df['r'] * df['redshift']).astype('float32')
        df['_g_/_redshift'] = (df['g'] / (df['redshift'] + eps)).astype('float32')
        df['_i_/_redshift'] = (df['i'] / (df['redshift'] + eps)).astype('float32')
        
    for a, b in color_pairs:
        if a in df.columns and b in df.columns:
            df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    if all(b in df.columns for b in ['g', 'r', 'i']):
        df['_balmer_break_proxy'] = ((df['g'] - df['r']) - (df['r'] - df['i'])).astype('float32')

    if all(b in df.columns for b in ['u', 'g', 'r', 'i', 'z']):
        df['g_r'] = df['g'] - df['r']
        df['r_i'] = df['r'] - df['i']
        df['u_g'] = df['u'] - df['g']
        df['i_z'] = df['i'] - df['z']
        
        df['_color_cross_ug_gr'] = (df['u_g'] * df['g_r']).astype('float32')
        df['_color_cross_gr_ri'] = (df['g_r'] * df['r_i']).astype('float32')
        
        df['stellar_locus_dist'] = np.sqrt((df['g_r'] - 0.52)**2 + (df['r_i'] - 0.25)**2).astype('float32')
        df['qso_locus_dist'] = np.sqrt((df['g_r'] - 0.24)**2 + (df['r_i'] - 0.15)**2).astype('float32')
        df['galaxy_locus_dist'] = np.sqrt((df['u_g'] - 1.50)**2 + (df['g_r'] - 0.70)**2).astype('float32')

    for col in cat_cols_init:
        print(col)
        if fit:
            df[col] = pd.Categorical(df[col])
            category_map[col] = df[col].dtype.categories
        else:
            df[col] = pd.Categorical(df[col], categories=category_map[col])

    for col in num_cols_init:
        cat_name = f"{col}_cat_"
        floored_values = np.floor(df[col])
        if fit:
            df[cat_name] = pd.Categorical(floored_values)
            category_map[col] = df[cat_name].dtype.categories
        else:
            df[cat_name] = pd.Categorical(floored_values, categories=category_map[col])

    for col, bins_list in {'delta': [100, 500]}.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)
                category_map[bin_name] = kb
            else:
                binned = category_map[bin_name].transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)

    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]: 
            combo_series = combo_series + '_' + df[col].astype(str)
        
        if fit:
            df[combo_name] = pd.Categorical(combo_series)
            category_map[combo_name] = df[combo_name].dtype.categories
        else:
            df[combo_name] = pd.Categorical(combo_series, categories=category_map[combo_name])

        df['spectral_rank_fe'] = df['spectral_type'].map({'M':1, 'G/K':2, 'A/F':3, 'O/B':4}).astype(int)
        df['population_rank_fe'] = df['galaxy_population'].map({'Red_Sequence':0,'Blue_Cloud':1}).astype(int)
        df['astro_score_fe'] = df['spectral_rank_fe'] * (df['population_rank_fe'] + 1)
        df['is_red_galaxy_candidate_fe'] = ((df['spectral_type']=='M') & (df['galaxy_population']=='Red_Sequence')).astype('int8')
        df['is_qso_candidate_fe'] = ((df['spectral_type'].isin(['A/F','O/B'])) & (df['galaxy_population']=='Blue_Cloud')).astype('int8')
        # df['_&_'.join([col + "_fe" for col in cat_cols_init])] = df[cat_cols_init].apply(lambda x: '_&_'.join(x.astype(str)), axis=1)
        # cat_cols_init.append('_&_'.join([col + "_fe" for col in cat_cols_init]))
        

        fltr_cols = ['u', 'g', 'r', 'i', 'z']
        # df['dom_fltr_fe'] = df[fltr_cols].idxmin(axis=1)
        # cat_cols_init.append('dom_fltr_fe')
        agg_map = { 'mean': 'mean','max': 'max','min': 'min','std': 'std','med': 'median'}
        for suffix, func in agg_map.items():
            df[f'{suffix}_fltr_fe'] = getattr(df[fltr_cols], func)(axis=1)

        for i in range(len(fltr_cols)):
            for j in range(i + 1, len(fltr_cols)):
                b1, b2 = fltr_cols[i], fltr_cols[j]
                df[f'{b1}_{b2}'] = df[b1] - df[b2]
        
        # Redshift interaction terms (multiplication) across all 10 color index bands
        df['u_g_redshift'] = df['u_g'] * df['redshift']
        df['g_r_redshift'] = df['g_r'] * df['redshift']
        df['r_i_redshift'] = df['r_i'] * df['redshift']
        df['i_z_redshift'] = df['i_z'] * df['redshift']
        df['u_r_redshift'] = df['u_r'] * df['redshift']
        df['g_i_redshift'] = df['g_i'] * df['redshift']
        df['r_z_redshift'] = df['r_z'] * df['redshift']
        df['u_i_redshift'] = df['u_i'] * df['redshift']
        df['g_z_redshift'] = df['g_z'] * df['redshift']
        df['u_z_redshift'] = df['u_z'] * df['redshift']

        # Color curvature differences (differences of differences)
        df['ug_gr'] = df['u_g'] - df['g_r']
        df['ug_ri'] = df['u_g'] - df['r_i']
        df['ug_iz'] = df['u_g'] - df['i_z']
        df['gr_ri'] = df['g_r'] - df['r_i']
        df['gr_iz'] = df['g_r'] - df['i_z']
        df['ri_iz'] = df['r_i'] - df['i_z']
        
        # Discretized stellar color temperature bins
        df['stellar_color_bin_fe'] = pd.cut(
            df['g_r'],
            bins=[-np.inf, 0.0, 0.4, 0.8, 1.2, np.inf],
            labels=[0, 1, 2, 3, 4]
        ).fillna(2).astype(int)

        # Equatorial coordinates -> Cartesian coordinates
        alpha_rad = np.radians(df['alpha'])
        delta_rad = np.radians(df['delta'])

        df['coord_x_fe'] = np.cos(delta_rad) * np.cos(alpha_rad)
        df['coord_y_fe'] = np.cos(delta_rad) * np.sin(alpha_rad)
        df['coord_z_fe'] = np.sin(delta_rad)

        # Redshift-scaled coordinates
        df['coord_x_redshift_fe'] = df['coord_x_fe'] * df['redshift']
        df['coord_y_redshift_fe'] = df['coord_y_fe'] * df['redshift']
        df['coord_z_redshift_fe'] = df['coord_z_fe'] * df['redshift']

        # Galactic coordinate transformation (J2000 NGP)
        x_gal = -0.05487554 * df['coord_x_fe'] - 0.87343710 * df['coord_y_fe'] - 0.48383499 * df['coord_z_fe']
        y_gal =  0.49410945 * df['coord_x_fe'] - 0.44482959 * df['coord_y_fe'] + 0.74698225 * df['coord_z_fe']
        z_gal = -0.86766614 * df['coord_x_fe'] - 0.19807639 * df['coord_y_fe'] + 0.45598380 * df['coord_z_fe']

        df['gal_x_fe'] = x_gal
        df['gal_y_fe'] = y_gal
        df['gal_z_fe'] = z_gal

    return df

# Feature Engineering Execution 

In [4]:
X_train = train.drop(columns=[ID, TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[ID])

X_train = feature_engineering(X_train, fit=True)
X_test = feature_engineering(X_test, fit=False)

SKEW_THRESHOLD = 1.5
UNIQUENESS_THRESHOLD = 15
current_num_cols = X_train.select_dtypes(exclude=['category', 'object']).columns.tolist()

for col in current_num_cols:
    if X_train[col].nunique() <= UNIQUENESS_THRESHOLD: continue
    if abs(X_train[col].skew()) > SKEW_THRESHOLD:
        X_train[col] = (np.sign(X_train[col]) * np.log1p(np.abs(X_train[col]))).astype('float32')
        X_test[col] = (np.sign(X_test[col]) * np.log1p(np.abs(X_test[col]))).astype('float32')

for col in X_train.columns.tolist():
    if isinstance(X_train[col].dtype, pd.CategoricalDtype) or X_train[col].dtype == 'object' or X_train[col].nunique() <= UNIQUENESS_THRESHOLD:
        X_train[col], X_test[col] = pd.Categorical(X_train[col]), pd.Categorical(X_test[col])

constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
if constant_cols:
    X_train.drop(columns=constant_cols, inplace=True)
    X_test.drop(columns=constant_cols, inplace=True)

spectral_type
galaxy_population
spectral_type
galaxy_population


## Data Processing

In [5]:
from sklearn.preprocessing import OrdinalEncoder

def ordinal_encode(train_df, test_df=None):
    train_df = train_df.copy()

    cat_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    if test_df is not None:
        test_df = test_df.copy()

        encoder.fit(
            pd.concat(
                [train_df[cat_cols], test_df[cat_cols]],
                axis=0
            )
        )

        train_df[cat_cols] = encoder.transform(train_df[cat_cols])
        test_df[cat_cols] = encoder.transform(test_df[cat_cols])

        return train_df, test_df, encoder

    encoder.fit(train_df[cat_cols])
    train_df[cat_cols] = encoder.transform(train_df[cat_cols])

    return train_df, encoder


train_df_enc, test_df_enc, encoder = ordinal_encode(X_train, X_test)
y_train = y_train.map(TARGET_MAPPING)

## Optuna Search

In [6]:
import lightgbm as lgb
import optuna
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

print(lgb.__version__)

X_train_, X_test_, y_train_, y_test_ = train_test_split(train_df_enc, y_train, test_size=0.2, random_state=42, stratify=y_train)

lgb_best_params = {
    'n_estimators': 1841,
    'learning_rate': 0.016121467631636684,
    'num_leaves': 126,
    'max_depth': 12,
    'min_child_samples': 7,
    'subsample': 0.9137011411523339,
    'colsample_bytree': 0.6472035788104408,
    'reg_alpha': 1.6143965568012781,
    'reg_lambda': 0.0457226857453295,
    'min_split_gain': 0.520062046419683,
    "max_bin": 63
}

def objective(trial):

    params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "device": "gpu",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            max(1200, int(lgb_best_params["n_estimators"] * 0.8)),
            min(2500, int(lgb_best_params["n_estimators"] * 1.2))
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            lgb_best_params["learning_rate"] * 0.7,
            lgb_best_params["learning_rate"] * 1.3,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            max(31, lgb_best_params["num_leaves"] - 20),
            min(255, lgb_best_params["num_leaves"] + 20)
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            max(6, lgb_best_params["max_depth"] - 2),
            lgb_best_params["max_depth"] + 2
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            max(2, lgb_best_params["min_child_samples"] - 3),
            lgb_best_params["min_child_samples"] + 3
        ),

        "subsample": trial.suggest_float(
            "subsample",
            max(0.7, lgb_best_params["subsample"] - 0.10),
            min(1.0, lgb_best_params["subsample"] + 0.05)
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            max(0.4, lgb_best_params["colsample_bytree"] - 0.10),
            min(1.0, lgb_best_params["colsample_bytree"] + 0.10)
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            max(1e-4, lgb_best_params["reg_alpha"] * 0.3),
            lgb_best_params["reg_alpha"] * 3,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            max(1e-5, lgb_best_params["reg_lambda"] * 0.2),
            lgb_best_params["reg_lambda"] * 5,
            log=True
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            max(0.0, lgb_best_params["min_split_gain"] - 0.30),
            lgb_best_params["min_split_gain"] + 0.30
        ),

        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "max_bin": 63
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(X_train_, y_train_)
    y_pred = model.predict(X_test_)
    b_acc = balanced_accuracy_score(y_test_, y_pred)
    print(f"Trial {trial.number} | BAL_ACC: {b_acc:.6f}")

    return b_acc


def best_trial_callback(study, trial):

    print("\n" + "=" * 60)
    print(f"Trial {trial.number} Completed")
    print(f"Trial BAL_ACC: {trial.value:.6f}")
    print(f"\nBest BAL_ACC So Far: {study.best_value:.6f}")
    print("\nBest Parameters So Far:")
    print(study.best_params)
    print("=" * 60)


if RUN_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=5,
        multivariate=True,
        group=True
    )

    study_lgb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="lightgbm_local_search"
    )

    study_lgb.enqueue_trial(lgb_best_params)

    study_lgb.optimize(
        objective,
        n_trials=100,
        callbacks=[best_trial_callback],
        show_progress_bar=True
    )

    print("\nBest BAL_ACC:")
    print(study_lgb.best_value)

    print("\nBest Parameters:")
    print(study_lgb.best_params)

4.6.0


## Best LightGBM 

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb

best_lgb_params = {'n_estimators': 1669, 'learning_rate': 0.019549762362923728, 'num_leaves': 122, 'max_depth': 13, 'min_child_samples': 8, 'subsample': 0.8764475135618286, 'colsample_bytree': 0.5890948761811168, 'reg_alpha': 3.8073608496396676, 'reg_lambda': 0.06650073755066985, 'min_split_gain': 0.25702982176749495}

best_lgb_params['class_weight']='balanced'
best_lgb_params['random_state']=42
best_lgb_params['max_bin']=63
best_lgb_params['objective']="multiclass"
best_lgb_params["metric"] = "multi_logloss"

if RUN_SKF:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []

    print(f"{'Fold':<6} {'Balanced Accuracy':^20}")
    print("-" * 30)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_, y_train_), 1):
        X_tr, X_val = X_train_.iloc[train_idx], X_train_.iloc[val_idx]
        y_tr, y_val = y_train_.iloc[train_idx], y_train_.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
        **best_lgb_params,
        verbose=-1
        )
        
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(y_val, y_pred)
        fold_scores.append(balanced_acc)
        
        print(f"{fold:<6} {balanced_acc:>20.4f}")

    print("-" * 30)
    print(f"{'Mean':<6} {np.mean(fold_scores):>20.4f}")
    print(f"{'Std':<6} {np.std(fold_scores):>20.4f}")

Fold    Balanced Accuracy  
------------------------------
1                    0.9662
2                    0.9661
3                    0.9639
4                    0.9643
5                    0.9645
------------------------------
Mean                 0.9650
Std                  0.0009


## Submission

In [8]:
lgb_model = lgb.LGBMClassifier(
    **best_lgb_params, 
    verbose=-1
    )
lgb_model.fit(train_df_enc, y_train)

FILE_NAME = "submission"
SUB_FILE_NAME = f"{FILE_NAME}.csv"

predictions = pd.Series(lgb_model.predict(test_df_enc)).map(TARGET_INV_MAPPING)
sub_df = pd.DataFrame({ID:test[ID], TARGET:predictions})
sub_df.to_csv(SUB_FILE_NAME, index=False)
sub_df.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
